# Multizone Model and Heat Transfer

This notebook explores the multizone combustion model and how heat transfer creates temperature stratification.

## Prerequisites

- `pip install -e .` for the base package
- For non-adiabatic multizone: `conda install -c conda-forge scikits.odes sundials`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from engine_sim.simulation.engine import EngineConfig, EngineSimulation
from engine_sim.engine.heat_transfer import HeatTransfer, WoschniParams
from engine_sim.engine.geometry import GeometryParams

## 1. Adiabatic vs Non-Adiabatic (Single Zone)

First, let's see the effect of heat transfer on a single-zone model.

In [ ]:
# Adiabatic
config_adi = EngineConfig.from_yaml()
config_adi.adiabatic = True
sim_adi = EngineSimulation(config_adi)
results_adi = sim_adi.run()

# Non-adiabatic
config_non = EngineConfig.from_yaml()
config_non.adiabatic = False
sim_non = EngineSimulation(config_non)
results_non = sim_non.run()

perf_adi = results_adi.calculate_performance()
perf_non = results_non.calculate_performance()

print(f"{'Metric':<20} {'Adiabatic':>12} {'Non-Adiabatic':>14}")
print('-' * 48)
print(f"{'IMEP [bar]':<20} {perf_adi['imep']:>12.2f} {perf_non['imep']:>14.2f}")
print(f"{'Peak Pressure [bar]':<20} {perf_adi['peak_pressure']:>12.1f} {perf_non['peak_pressure']:>14.1f}")
print(f"{'Peak Temp [K]':<20} {perf_adi['peak_temperature']:>12.0f} {perf_non['peak_temperature']:>14.0f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(results_adi.crank_angle, results_adi.temperature, 'r--', label='Adiabatic', linewidth=2)
ax1.plot(results_non.crank_angle, results_non.temperature, 'b-', label='Non-Adiabatic', linewidth=2)
ax1.set_xlabel('Crank Angle [deg]')
ax1.set_ylabel('Temperature [K]')
ax1.set_title('Temperature: Adiabatic vs Non-Adiabatic')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(results_adi.crank_angle, results_adi.pressure / 1e5, 'r--', label='Adiabatic', linewidth=2)
ax2.plot(results_non.crank_angle, results_non.pressure / 1e5, 'b-', label='Non-Adiabatic', linewidth=2)
ax2.set_xlabel('Crank Angle [deg]')
ax2.set_ylabel('Pressure [bar]')
ax2.set_title('Pressure: Adiabatic vs Non-Adiabatic')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. The Woschni Heat Transfer Model

The heat transfer rate is:

$$\dot{Q} = h \cdot A_{total} \cdot (T_{gas} - T_{wall})$$

where $h$ is the convective heat transfer coefficient from the Woschni correlation.

Let's visualize how $h$ and $\dot{Q}$ vary over the cycle.

In [ ]:
geom = GeometryParams(bore=0.086, stroke=0.086, con_rod=0.1455, comp_ratio=12.5)
ht = HeatTransfer(geom=geom, params=WoschniParams())

# Use results from non-adiabatic simulation to compute heat transfer along the cycle
ca_rad = np.deg2rad(results_non.crank_angle)
rpm = 2000.0
T_wall = 358.15  # K
P_ref = results_non.pressure[0]
T_ref = results_non.temperature[0]
V_ref = results_non.volume[0]

Q_array = []
h_array = []
for i in range(len(ca_rad)):
    gamma = 1.35
    p_mot = P_ref * (V_ref / results_non.volume[i])**gamma
    Q, h = ht.heat_transfer_rate(
        ca_rad[i], rpm, results_non.pressure[i], results_non.temperature[i],
        p_mot, P_ref, T_ref, V_ref, T_wall
    )
    Q_array.append(Q)
    h_array.append(h)

Q_array = np.array(Q_array)
h_array = np.array(h_array)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax1.plot(results_non.crank_angle, h_array, 'b-', linewidth=2)
ax1.set_ylabel('h [W/(m2 K)]')
ax1.set_title('Woschni Heat Transfer Coefficient')
ax1.grid(True, alpha=0.3)

ax2.plot(results_non.crank_angle, Q_array / 1000, 'r-', linewidth=2)
ax2.set_xlabel('Crank Angle [deg]')
ax2.set_ylabel('Heat Transfer Rate [kW]')
ax2.set_title('Wall Heat Loss Rate')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Peak heat transfer coefficient: {np.max(h_array):.0f} W/(m2 K)")
print(f"Peak heat transfer rate: {np.max(Q_array)/1000:.1f} kW")

## 3. Multizone Model: Zone Area Fractions

In the multizone model, the cylinder is divided into concentric zones from core to wall. Each zone receives a different fraction of the total heat transfer based on its proximity to the wall:

$$A_i = \frac{3(2(i-1))^2}{2 N_{zones}(N_{zones}-1)(2N_{zones}-1)}$$

- Zone 1 (core): $A_i = 0$ (no wall contact)
- Zone N (wall): maximum $A_i$ (most heat loss)

In [ ]:
def area_fractions(nzones):
    Ai = np.zeros(nzones)
    for i in range(nzones):
        if nzones > 1:
            Ai[i] = 3 * (2 * (i - 1))**2 / (2 * nzones * (nzones - 1) * (2 * nzones - 1))
        else:
            Ai[i] = 1.0
    return Ai

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, nz in zip(axes, [5, 10, 20]):
    Ai = area_fractions(nz)
    colors = plt.cm.plasma(np.linspace(0.1, 0.9, nz))
    ax.bar(range(1, nz+1), Ai, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xlabel('Zone Number')
    ax.set_ylabel('Area Fraction')
    ax.set_title(f'{nz} Zones (sum={np.sum(Ai):.3f})')
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 4. Multizone Simulation (Non-Adiabatic with CVODE)

Now let's run a multizone simulation and observe the temperature stratification.

**Note:** This requires `scikits.odes` (CVODE solver) for the non-adiabatic case. If not installed, this cell will fail.

In [ ]:
config_mz = EngineConfig.from_yaml()
config_mz.model_type = "multi"
config_mz.nzones = 10
config_mz.adiabatic = False
config_mz.method = "CVODE"
config_mz.rtol = 1e-5
config_mz.atol = 1e-12

sim_mz = EngineSimulation(config_mz)
y0 = sim_mz.setup_initial_state()
print(f"State vector size: {len(y0)} variables")
print(f"  3 bulk + {config_mz.nzones} zones x 34 vars + 33 bulk species")

sol = sim_mz.solver.solve_closed_cycle(
    y0=y0, rpm=config_mz.speed, T_wall=config_mz.wall_temp,
    ca_start=config_mz.start_ca, ca_end=config_mz.end_ca
)

In [ ]:
from engine_sim.simulation.results import SimulationResults

results_mz = SimulationResults.from_solver_output(
    sol, sim_mz.chemistry.gas.species_names,
    model_type='multi', mechanism=config_mz.mechanism
)

# Extract zone temperatures
nsp = len(sim_mz.chemistry.gas.species_names)
nzones = config_mz.nzones
zone_temps = np.zeros((nzones, len(results_mz.crank_angle)))
for i in range(nzones):
    zone_temps[i, :] = sol['y'][3 + i * (nsp + 1), :]

# Plot zone temperatures
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.plasma(np.linspace(0, 1, nzones))
for i in range(nzones):
    label = f'Zone {i+1}'
    if i == 0:
        label += ' (core)'
    elif i == nzones - 1:
        label += ' (wall)'
    ax1.plot(results_mz.crank_angle, zone_temps[i, :], color=colors[i],
             linewidth=1.5, label=label)

ax1.set_xlabel('Crank Angle [deg]')
ax1.set_ylabel('Temperature [K]')
ax1.set_title(f'{nzones}-Zone Temperatures (Non-Adiabatic)')
ax1.legend(fontsize=7, ncol=2)
ax1.grid(True, alpha=0.3)

# Stratification
stratification = zone_temps[0, :] - zone_temps[-1, :]
ax2.plot(results_mz.crank_angle, stratification, 'g-', linewidth=2)
ax2.set_xlabel('Crank Angle [deg]')
ax2.set_ylabel('T_core - T_wall [K]')
ax2.set_title('Temperature Stratification')
ax2.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax2.grid(True, alpha=0.3)

max_idx = np.argmax(np.abs(stratification))
ax2.annotate(f'Max: {stratification[max_idx]:.0f} K',
             xy=(results_mz.crank_angle[max_idx], stratification[max_idx]),
             xytext=(10, 10), textcoords='offset points',
             bbox=dict(boxstyle='round', fc='yellow', alpha=0.7),
             arrowprops=dict(arrowstyle='->'))

plt.tight_layout()
plt.show()

# Performance
perf_mz = results_mz.calculate_performance()
print(f"\nMultizone Performance:")
print(f"  IMEP:          {perf_mz['imep']:.2f} bar")
print(f"  Peak Pressure: {perf_mz['peak_pressure']:.1f} bar")
print(f"  Peak Temp:     {perf_mz['peak_temperature']:.0f} K")
print(f"  Max Stratification: {np.max(np.abs(stratification)):.0f} K")

## 5. Effect of Wall Temperature

The wall temperature directly affects the heat loss rate and therefore the temperature stratification.

In [ ]:
T_wall_values = [350, 400, 450]

fig, ax = plt.subplots(figsize=(10, 6))

for T_w in T_wall_values:
    cfg = EngineConfig.from_yaml()
    cfg.wall_temp = float(T_w)
    cfg.adiabatic = False
    s = EngineSimulation(cfg)
    r = s.run()
    ax.plot(r.crank_angle, r.temperature, linewidth=2, label=f'T_wall = {T_w} K')

ax.set_xlabel('Crank Angle [deg]')
ax.set_ylabel('Temperature [K]')
ax.set_title('Effect of Wall Temperature (Single Zone, Non-Adiabatic)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()